# 实战案例：基于注意力的图像描述 (PyTorch CPU版)

本notebook将PaddlePaddle版本的ARCTIC模型转为PyTorch实现，默认在CPU上运行。

## 环境准备

需要安装以下依赖：

In [ ]:
# 安装依赖
# !pip install torch torchvision nltk pillow matplotlib
# 解压数据
# !unzip -q ./data/data243982/flickr8k.zip -d ./data/

## 读取数据

### 整理数据集

构建词典，将文本描述转为向量，记录图像路径。

In [ ]:
%matplotlib inline
import os
from os.path import join as pjoin
import json
import random
from collections import defaultdict, Counter
from PIL import Image
from matplotlib import pyplot as plt

def create_dataset(data_dir='./data',
                   dataset='flickr8k',
                   captions_per_image=5,
                   min_word_count=5,
                   max_len=30):
    """
    参数：
        data_dir: 数据存储目录
        dataset：数据集名称
        captions_per_image：每张图片对应的文本描述数
        min_word_count：仅考虑在数据集中（除测试集外）出现5次的词
        max_len：文本描述包含的最大单词数
    输出：
        vocab.json, train_data.json, val_data.json, test_data.json
    """
    karpathy_json_path = pjoin(data_dir, '%s/dataset_flickr8k.json' % dataset)
    image_folder = pjoin(data_dir, '%s/images' % dataset)
    output_folder = pjoin(data_dir, '%s' % dataset)

    with open(karpathy_json_path, 'r') as j:
        data = json.load(j)

    image_paths = defaultdict(list)
    image_captions = defaultdict(list)
    vocab = Counter()

    for img in data['images']:
        split = img['split']
        captions = []
        for c in img['sentences']:
            if split != 'test':
                vocab.update(c['tokens'])
            if len(c['tokens']) <= max_len:
                captions.append(c['tokens'])
        if len(captions) == 0:
            continue
        path = os.path.join(image_folder, img['filename'])
        image_paths[split].append(path)
        image_captions[split].append(captions)

    words = [w for w in vocab.keys() if vocab[w] > min_word_count]
    vocab = {k: v + 1 for v, k in enumerate(words)}
    vocab['<pad>'] = 0
    vocab['<unk>'] = len(vocab)
    vocab['<start>'] = len(vocab)
    vocab['<end>'] = len(vocab)

    with open(os.path.join(output_folder, 'vocab.json'), 'w') as fw:
        json.dump(vocab, fw)

    for split in image_paths:
        imgpaths = image_paths[split]
        imcaps = image_captions[split]
        enc_captions = []
        for i, path in enumerate(imgpaths):
            img = Image.open(path)
            if len(imcaps[i]) < captions_per_image:
                captions = imcaps[i] + \
                    [random.choice(imcaps[i]) for _ in range(captions_per_image - len(imcaps[i]))]
            else:
                captions = random.sample(imcaps[i], k=captions_per_image)
            assert len(captions) == captions_per_image
            for j, c in enumerate(captions):
                enc_c = [vocab['<start>']] + [vocab.get(word, vocab['<unk>']) for word in c] + [vocab['<end>']]
                enc_captions.append(enc_c)
        assert len(imgpaths) * captions_per_image == len(enc_captions)
        data = {'IMAGES': imgpaths, 'CAPTIONS': enc_captions}
        with open(pjoin(output_folder, split + '_data.json'), 'w') as fw:
            json.dump(data, fw)

data_dir = './data'
if not os.path.exists(pjoin(data_dir, 'flickr8k', 'vocab.json')):
    create_dataset(data_dir)

### 定义数据集类

**关键改动**：`paddle.io.Dataset` → `torch.utils.data.Dataset`，`paddle.to_tensor` → `torch.tensor`

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class ImageTextDataset(Dataset):
    def __init__(self, dataset_path, vocab_path, split, captions_per_image=5, max_len=30, transform=None):
        self.split = split
        assert self.split in {'train', 'val', 'test'}
        self.cpi = captions_per_image
        self.max_len = max_len

        with open(dataset_path, 'r') as f:
            self.data = json.load(f)
        with open(vocab_path, 'r') as f:
            self.vocab = json.load(f)

        self.transform = transform
        self.dataset_size = len(self.data['CAPTIONS'])

    def __getitem__(self, i):
        img = Image.open(self.data['IMAGES'][i // self.cpi]).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)

        caplen = len(self.data['CAPTIONS'][i])
        # 补齐到固定长度
        caption = torch.tensor(
            self.data['CAPTIONS'][i] + [self.vocab['<pad>']] * (self.max_len + 2 - caplen),
            dtype=torch.long
        )
        return img, caption, caplen

    def __len__(self):
        return self.dataset_size

### 批量读取数据

**关键改动**：`paddle.io.DataLoader` → `torch.utils.data.DataLoader`，图像预处理使用 `torchvision.transforms`

In [ ]:
def mktrainval(data_dir, vocab_path, batch_size, workers=0):
    train_tx = transforms.Compose([
        transforms.Resize(256),
        transforms.RandomCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    val_tx = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    train_set = ImageTextDataset(os.path.join(data_dir, 'train_data.json'),
                                vocab_path, 'train', transform=train_tx)
    valid_set = ImageTextDataset(os.path.join(data_dir, 'val_data.json'),
                                vocab_path, 'val', transform=val_tx)
    test_set = ImageTextDataset(os.path.join(data_dir, 'test_data.json'),
                                vocab_path, 'test', transform=val_tx)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=workers)
    valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False, num_workers=workers, drop_last=False)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=workers, drop_last=False)

    return train_loader, valid_loader, test_loader

## 定义模型

### 图像编码器

**关键改动**：
- `paddle.vision.models` → `torchvision.models`
- `nn.Layer` → `nn.Module`
- `param.requires_grad = finetuned` 用法不变（PyTorch原生支持）
- `model.children()` 用法一致

In [ ]:
import torch.nn as nn
import torchvision.models as models

class ImageEncoder(nn.Module):
    def __init__(self, finetuned=True):
        super(ImageEncoder, self).__init__()
        # 使用预训练的ResNet-101，去掉最后两层（avgpool + fc）
        model = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        self.grid_representation_extractor = nn.Sequential(*(list(model.children())[:-2]))
        for param in self.grid_representation_extractor.parameters():
            param.requires_grad = finetuned

    def forward(self, images):
        # images: (batch, 3, 224, 224)
        # out: (batch, 2048, 7, 7)
        out = self.grid_representation_extractor(images)
        return out

### 注意力机制（加性注意力）

**关键改动**：
- `nn.Layer` → `nn.Module`
- `paddle.bmm` → `torch.bmm`
- `axis` → `dim`（PyTorch用`dim`关键字）

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, query_dim, key_dim, attn_dim):
        super(AdditiveAttention, self).__init__()
        self.attn_w_1_q = nn.Linear(query_dim, attn_dim)
        self.attn_w_1_k = nn.Linear(key_dim, attn_dim)
        self.attn_w_2 = nn.Linear(attn_dim, 1)
        self.tanh = nn.Tanh()
        self.softmax = nn.Softmax(dim=1)  # Paddle: axis=1 → PyTorch: dim=1

    def forward(self, query, key_value):
        """
        query: (batch_size, q_dim)
        key_value: (batch_size, n_kv, kv_dim)
        """
        # (batch_size, 1, attn_dim)
        queries = self.attn_w_1_q(query).unsqueeze(1)
        # (batch_size, n_kv, attn_dim)
        keys = self.attn_w_1_k(key_value)
        # (batch_size, n_kv)
        attn = self.attn_w_2(self.tanh(queries + keys)).squeeze(2)
        attn = self.softmax(attn)
        # (batch_size, 1, kv_dim)
        output = torch.bmm(attn.unsqueeze(1), key_value).squeeze(1)
        return output, attn

### 文本解码器

**关键改动**：
- `nn.Layer` → `nn.Module`
- `nn.initializer.Uniform(low=-0.1, high=0.1)` → 手动 `nn.init.uniform_`
- Paddle GRU的 `time_major=True` → PyTorch GRU的 `batch_first=False`（默认）
- `paddle.concat` → `torch.cat`
- `paddle.zeros` → `torch.zeros`
- `paddle.to_tensor` → `torch.tensor`
- `.numpy()` 用法不变
- `(-cap_lens).argsort(axis=0)` → `cap_lens.argsort(descending=True)`

In [ ]:
class AttentionDecoder(nn.Module):
    def __init__(self, image_code_dim, vocab_size, word_dim, attention_dim, hidden_size, num_layers, dropout=0.5):
        super(AttentionDecoder, self).__init__()
        self.embed = nn.Embedding(vocab_size, word_dim)
        nn.init.uniform_(self.embed.weight, -0.1, 0.1)

        self.attention = AdditiveAttention(hidden_size, image_code_dim, attention_dim)
        self.init_state = nn.Linear(image_code_dim, num_layers * hidden_size)
        nn.init.uniform_(self.init_state.weight, -0.1, 0.1)

        # PyTorch GRU: batch_first=False（输入shape为 (seq_len, batch, input_size)）
        self.rnn = nn.GRU(word_dim + image_code_dim, hidden_size, num_layers)
        self.dropout = nn.Dropout(p=dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)
        nn.init.uniform_(self.fc.weight, -0.1, 0.1)

        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.vocab_size = vocab_size

    def init_hidden_state(self, image_code, captions, cap_lens):
        batch_size, image_code_dim = image_code.shape[0], image_code.shape[1]
        # (batch, C, H, W) -> (batch, H*W, C)
        image_code = image_code.permute(0, 2, 3, 1)
        image_code = image_code.reshape(batch_size, -1, image_code_dim)

        # 按caption长度从长到短排序
        sorted_cap_indices = cap_lens.argsort(descending=True)
        sorted_cap_lens = cap_lens[sorted_cap_indices]
        captions = captions[sorted_cap_indices]
        image_code = image_code[sorted_cap_indices]

        if captions.dim() == 1:
            captions = captions.unsqueeze(0)
        if image_code.dim() == 2:
            image_code = image_code.unsqueeze(0)

        # 初始化隐状态
        hidden_state = self.init_state(image_code.mean(dim=1))
        # (batch, num_layers * hidden) -> (num_layers, batch, hidden)
        hidden_state = hidden_state.reshape(batch_size, self.num_layers, self.hidden_size).permute(1, 0, 2).contiguous()
        return image_code, captions, sorted_cap_lens, sorted_cap_indices, hidden_state

    def forward_step(self, image_code, curr_cap_embed, hidden_state):
        # 注意力
        context, alpha = self.attention(hidden_state[-1], image_code)
        # 拼接上下文和当前词嵌入，增加seq_len维度
        x = torch.cat((context, curr_cap_embed), dim=-1).unsqueeze(0)  # (1, batch, dim)
        out, hidden_state = self.rnn(x, hidden_state)
        preds = self.fc(self.dropout(out.squeeze(0)))
        return preds, alpha, hidden_state

    def forward(self, image_code, captions, cap_lens):
        image_code, captions, sorted_cap_lens, sorted_cap_indices, hidden_state = \
            self.init_hidden_state(image_code, captions, cap_lens)
        batch_size = image_code.shape[0]
        lengths = sorted_cap_lens.numpy() - 1

        predictions = torch.zeros(batch_size, lengths[0], self.vocab_size)
        alphas = torch.zeros(batch_size, lengths[0], image_code.shape[1])

        cap_embeds = self.embed(captions)

        for step in range(lengths[0]):
            real_batch_size = np.where(lengths > step)[0].shape[0]
            preds, alpha, hidden_state = self.forward_step(
                image_code[:real_batch_size],
                cap_embeds[:real_batch_size, step, :],
                hidden_state[:, :real_batch_size, :].contiguous()
            )
            predictions[:real_batch_size, step, :] = preds
            alphas[:real_batch_size, step, :] = alpha

        return predictions, alphas, captions, lengths, sorted_cap_indices

### ARCTIC模型

**关键改动**：
- `nn.Layer` → `nn.Module`
- `paddle.full` → `torch.full`
- `repeat_interleave` / `expand` 用法一致
- `nn.functional.log_softmax` → `torch.nn.functional.log_softmax`
- `paddle.take_along_axis` → `torch.index_select` / 手动索引

In [ ]:
import torch.nn.functional as F

class ARCTIC(nn.Module):
    def __init__(self, image_code_dim, vocab, word_dim, attention_dim, hidden_size, num_layers):
        super(ARCTIC, self).__init__()
        self.vocab = vocab
        self.encoder = ImageEncoder()
        self.decoder = AttentionDecoder(image_code_dim, len(vocab), word_dim, attention_dim, hidden_size, num_layers)

    def forward(self, images, captions, cap_lens):
        image_code = self.encoder(images)
        return self.decoder(image_code, captions, cap_lens)

    def generate_by_beamsearch(self, images, beam_k, max_len):
        vocab_size = len(self.vocab)
        image_codes = self.encoder(images)
        texts = []

        for image_code in image_codes:
            image_code = image_code.unsqueeze(0).repeat_interleave(beam_k, dim=0)
            cur_sents = torch.full((beam_k, 1), self.vocab['<start>'], dtype=torch.long)
            cur_sent_embed = self.decoder.embed(cur_sents)[:, 0, :]
            sent_lens = torch.ones(beam_k, dtype=torch.long)
            image_code, cur_sent_embed, _, _, hidden_state = \
                self.decoder.init_hidden_state(image_code, cur_sent_embed, sent_lens)

            end_sents = []
            end_probs = []
            probs = torch.zeros(beam_k, 1)
            k = beam_k

            while True:
                preds, _, hidden_state = self.decoder.forward_step(
                    image_code[:k], cur_sent_embed, hidden_state)
                preds = F.log_softmax(preds, dim=1)
                probs = probs.expand_as(preds) + preds

                if cur_sents.shape[1] == 1:
                    values, indices = probs[0].topk(k, dim=0, largest=True, sorted=True)
                else:
                    values, indices = probs.reshape(-1).topk(k, dim=0, largest=True, sorted=True)

                sent_indices = indices // vocab_size
                word_indices = indices % vocab_size

                cur_sents = cur_sents[sent_indices]
                if cur_sents.dim() < 2:
                    cur_sents = cur_sents.unsqueeze(0)
                cur_sents = torch.cat([cur_sents, word_indices.unsqueeze(1)], dim=1)

                end_indices = [idx for idx, word in enumerate(word_indices) if word == self.vocab['<end>']]
                if len(end_indices) > 0:
                    end_probs.extend(values[end_indices].tolist())
                    end_sents.extend(cur_sents[end_indices].tolist())
                    k -= len(end_indices)
                    if k == 0:
                        break

                cur_indices = [idx for idx, word in enumerate(word_indices) if word != self.vocab['<end>']]
                if len(cur_indices) > 0:
                    cur_sent_indices = sent_indices[cur_indices]
                    cur_word_indices = word_indices[cur_indices]
                    cur_sents = cur_sents[cur_indices]
                    probs = values[cur_indices].reshape(-1, 1)
                    hidden_state = hidden_state[:, cur_sent_indices, :].contiguous()
                    cur_sent_embed = self.decoder.embed(cur_word_indices.reshape(-1, 1))[:, 0, :]

                if cur_sents.shape[1] >= max_len:
                    break

            if len(end_sents) == 0:
                gen_sent = cur_sents[0].tolist()
            else:
                gen_sent = end_sents[end_probs.index(max(end_probs))]
            texts.append(gen_sent)
        return texts

## 定义损失函数

**关键改动**：`nn.Layer` → `nn.Module`，`paddle.concat` → `torch.cat`

In [ ]:
class CrossEntropyLoss(nn.Module):
    def __init__(self):
        super(CrossEntropyLoss, self).__init__()
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, predictions, targets, lengths):
        preds = []
        gts = []
        for i in range(len(lengths)):
            preds.append(predictions[i, :lengths[i], :])
            gts.append(targets[i, :lengths[i]])
        preds = torch.cat(preds, dim=0)
        gts = torch.cat(gts, dim=0)
        return self.loss_fn(preds, gts)

## 选择优化方法

**关键改动**：`paddle.optimizer.Adam` → `torch.optim.Adam`

In [ ]:
def get_optimizer(model, config):
    return torch.optim.Adam(model.parameters(), lr=config.learning_rate)

## 评估指标 (BLEU-4)

**关键改动**：`paddle.no_grad()` → `torch.no_grad()`

In [ ]:
from nltk.translate.bleu_score import corpus_bleu

def filter_useless_words(sent, filterd_words):
    return [w for w in sent if w not in filterd_words]

def evaluate(data_loader, model, config):
    model.eval()
    cands = []
    refs = []
    filterd_words = set({model.vocab['<start>'], model.vocab['<end>'], model.vocab['<pad>']})
    cpi = config.captions_per_image

    for i, (imgs, caps, caplens) in enumerate(data_loader):
        with torch.no_grad():
            texts = model.generate_by_beamsearch(imgs, config.beam_k, config.max_len + 2)
            cands.extend([filter_useless_words(text, filterd_words) for text in texts])
            refs.extend([filter_useless_words(cap, filterd_words) for cap in caps.tolist()])

    multiple_refs = []
    for idx in range(len(refs)):
        multiple_refs.append(refs[(idx // cpi) * cpi : (idx // cpi) * cpi + cpi])

    bleu4 = corpus_bleu(multiple_refs, cands, weights=(0.25, 0.25, 0.25, 0.25))
    model.train()
    return bleu4

## 训练模型

**关键改动**：
- **设备**：显式使用 `device = torch.device('cpu')`
- `optimizer.clear_grad()` → `optimizer.zero_grad()`
- `nn.utils.clip_grad_norm_` 用法一致（PyTorch原生支持）
- `paddle.save` → `torch.save` / `torch.load`
- `loss.item()` 用法一致
- `model.state_dict()` / `optimizer.state_dict()` 用法一致

**注意**：CPU训练会比较慢，建议减小 `num_epochs` 和 `batch_size` 做验证。

In [ ]:
from argparse import Namespace

# ========== 超参数 ==========
config = Namespace(
    max_len=30,
    captions_per_image=5,
    batch_size=32,       # CPU上可适当减小，如8或16
    image_code_dim=2048,
    word_dim=512,
    hidden_size=512,
    attention_dim=512,
    num_layers=1,
    learning_rate=0.0005,
    num_epochs=10,
    grad_clip=5.0,
    alpha_weight=1.0,
    evaluate_step=900,
    checkpoint=None,
    best_checkpoint='model/ARCTIC/best_flickr8k.ckpt',
    last_checkpoint='model/ARCTIC/last_flickr8k.ckpt',
    beam_k=5
)

# ========== CPU设备 ==========
device = torch.device('cpu')
print(f'使用设备: {device}')

# ========== 数据 ==========
data_dir = './data'
vocab_path = pjoin(data_dir, 'flickr8k/vocab.json')
train_loader, valid_loader, test_loader = mktrainval(
    pjoin(data_dir, 'flickr8k'), vocab_path, config.batch_size)

with open(vocab_path, 'r') as f:
    vocab = json.load(f)

# ========== 模型 ==========
start_epoch = 0
checkpoint = config.checkpoint

if checkpoint is None:
    model = ARCTIC(config.image_code_dim, vocab, config.word_dim,
                   config.attention_dim, config.hidden_size, config.num_layers)
else:
    ckpt = torch.load(checkpoint, map_location=device)
    start_epoch = ckpt['epoch'] + 1
    model = ARCTIC(config.image_code_dim, vocab, config.word_dim,
                   config.attention_dim, config.hidden_size, config.num_layers)
    model.load_state_dict(ckpt['model'])

model = model.to(device)

# ========== 优化器和损失 ==========
optimizer = get_optimizer(model, config)
loss_fn = CrossEntropyLoss()

os.makedirs(os.path.dirname(config.best_checkpoint), exist_ok=True)

model.train()
best_res = 0
print("开始训练")

for epoch in range(start_epoch, config.num_epochs):
    for i, (imgs, caps, caplens) in enumerate(train_loader):
        imgs = imgs.to(device)
        caps = caps.to(device)
        caplens = torch.tensor(caplens, dtype=torch.long)

        optimizer.zero_grad()  # Paddle: clear_grad()

        # 前馈
        predictions, alphas, sorted_captions, lengths, sorted_cap_indices = model(imgs, caps, caplens)

        # 损失
        loss = loss_fn(predictions, sorted_captions[:, 1:], lengths)
        loss += config.alpha_weight * ((1. - alphas.sum(dim=1)) ** 2).mean()

        loss.backward()

        if config.grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)

        optimizer.step()

        if (i + 1) % 100 == 0:
            print(f'epoch {epoch}, step {i+1}: loss={loss.item():.2f}')

        if (i + 1) % config.evaluate_step == 0:
            bleu_score = evaluate(valid_loader, model, config)
            state = {
                'epoch': epoch,
                'step': i,
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict()
            }
            if best_res < bleu_score:
                best_res = bleu_score
                torch.save(state, config.best_checkpoint)
            torch.save(state, config.last_checkpoint)
            print(f'Validation@epoch {epoch}, step {i+1}, BLEU-4={bleu_score:.4f}')

# ========== 测试 ==========
ckpt = torch.load(config.best_checkpoint, map_location=device)
model.load_state_dict(ckpt['model'])
bleu_score = evaluate(test_loader, model, config)
print(f'Test BLEU-4={bleu_score:.4f} (best epoch={ckpt["epoch"]})')

## Paddle → PyTorch 对照速查表

| Paddle | PyTorch | 说明 |
|--------|---------|------|
| `nn.Layer` | `nn.Module` | 模型基类 |
| `paddle.to_tensor(x, dtype='int64')` | `torch.tensor(x, dtype=torch.long)` | 创建张量 |
| `paddle.concat(tensors, axis=0)` | `torch.cat(tensors, dim=0)` | 拼接 |
| `paddle.zeros(shape)` | `torch.zeros(shape)` | 全零张量 |
| `paddle.full(shape, val)` | `torch.full(shape, val)` | 填充张量 |
| `paddle.bmm(a, b)` | `torch.bmm(a, b)` | 批矩阵乘 |
| `tensor.transpose((0,2,3,1))` | `tensor.permute(0,2,3,1)` | 维度变换 |
| `Softmax(axis=1)` | `Softmax(dim=1)` | axis→dim |
| `optimizer.clear_grad()` | `optimizer.zero_grad()` | 清除梯度 |
| `paddle.save(obj, path)` | `torch.save(obj, path)` | 保存 |
| `paddle.load(path)` | `torch.load(path, map_location=device)` | 加载 |
| `nn.initializer.Uniform(low, high)` | `nn.init.uniform_(tensor, low, high)` | 初始化 |
| `paddle.device.set_device('gpu:0')` | `device = torch.device('cpu')` | 设备选择 |
| `paddle.io.Dataset` | `torch.utils.data.Dataset` | 数据集基类 |
| `paddle.io.DataLoader` | `torch.utils.data.DataLoader` | 数据加载器 |
| `paddle.vision.transforms` | `torchvision.transforms` | 图像变换 |
| `paddle.vision.models` | `torchvision.models` | 预训练模型 |
| GRU `time_major=True` | GRU `batch_first=False`(默认) | 时序维度在前 |